In [29]:
# Cell 1: Imports and Setup
import mutalyzer
import pandas as pd
import numpy as np
import hgvs.parser
import hgvs.dataproviders.uta
import hgvs.assemblymapper
import sys
import gc
import os
import pickle
import time
import json
import warnings
from collections import defaultdict
from IPython.display import clear_output, display, HTML
import threading

warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)



In [30]:
# Cell 2: Fixed HGVS Processor Class Definition
class HGVSProcessor:
    """HGVS Processor with resume capability for Jupyter Notebook"""
    
    def __init__(self, checkpoint_dir='./hgvs_checkpoints'):
        self.checkpoint_dir = checkpoint_dir
        os.makedirs(checkpoint_dir, exist_ok=True)
        
        # File paths
        self.checkpoint_file = os.path.join(checkpoint_dir, 'processing_checkpoint.json')
        self.cache_file = os.path.join(checkpoint_dir, 'variant_cache.pkl')
        self.progress_file = os.path.join(checkpoint_dir, 'processing_progress.log')
        self.results_file = os.path.join(checkpoint_dir, 'partial_results.pkl')
        
        # State variables
        self.variant_cache = {}
        self.checkpoint = None
        self.is_interrupted = False
        
        # Initialize HGVS components
        self.hp, self.hdp, self.am = self._initialize_hgvs()
        
        # Load existing state
        self._load_state()
        
        print(f"✓ HGVS Processor initialized")
        print(f"  Checkpoint directory: {checkpoint_dir}")
        print(f"  Cache size: {len(self.variant_cache):,} variants")
        
    def _initialize_hgvs(self, max_retries=5):
        """Initialize HGVS components with retry logic"""
        for attempt in range(max_retries):
            try:
                print(f"Initializing HGVS components (attempt {attempt + 1}/{max_retries})...")
                hp = hgvs.parser.Parser()
                hdp = hgvs.dataproviders.uta.connect()
                am = hgvs.assemblymapper.AssemblyMapper(
                    hdp, 
                    assembly_name='GRCh38', 
                    alt_aln_method='splign'
                )
                print("✓ HGVS components initialized successfully")
                return hp, hdp, am
                
            except Exception as e:
                if attempt < max_retries - 1:
                    wait_time = 2 ** attempt  # Exponential backoff
                    print(f"  Connection failed: {e}")
                    print(f"  Retrying in {wait_time} seconds...")
                    time.sleep(wait_time)
                else:
                    print(f"✗ Failed to initialize HGVS after {max_retries} attempts")
                    raise
    
    def _load_state(self):
        """Load checkpoint, cache, and results"""
        # Load cache
        if os.path.exists(self.cache_file):
            try:
                with open(self.cache_file, 'rb') as f:
                    self.variant_cache = pickle.load(f)
                print(f"✓ Loaded {len(self.variant_cache):,} cached variants")
            except Exception as e:
                print(f"✗ Error loading cache: {e}")
                self.variant_cache = {}
        
        # Load checkpoint
        if os.path.exists(self.checkpoint_file):
            try:
                with open(self.checkpoint_file, 'r') as f:
                    self.checkpoint = json.load(f)
                print(f"✓ Found checkpoint from: {self.checkpoint.get('timestamp', 'unknown')}")
                print(f"  Phase: {self.checkpoint.get('phase', 1)}")
                print(f"  Processed unique variants: {len(self.checkpoint.get('processed_unique', [])):,}")
                print(f"  Processed rows: {len(self.checkpoint.get('processed_rows', [])):,}")
                print(f"  Progress: {self.checkpoint.get('progress_percent', 0):.1f}%")
            except Exception as e:
                print(f"✗ Error loading checkpoint: {e}")
                self.checkpoint = None
        
        # Load partial results if any
        if os.path.exists(self.results_file):
            try:
                with open(self.results_file, 'rb') as f:
                    self.partial_results = pickle.load(f)
                print(f"✓ Loaded partial results for {len(self.partial_results):,} rows")
            except:
                self.partial_results = {}
    
    def _save_checkpoint(self, phase, processed_unique_count, processed_rows_count, 
                        total_unique, total_rows, current_variant=None):
        """Save current processing state - FIXED VERSION"""
        try:
            # Calculate progress percentage safely
            if total_rows > 0:
                progress_percent = (processed_rows_count / total_rows) * 100
            else:
                progress_percent = 0
            
            checkpoint = {
                'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
                'phase': phase,
                'processed_unique_count': processed_unique_count,
                'processed_rows_count': processed_rows_count,
                'total_unique': total_unique,
                'total_rows': total_rows,
                'current_variant': current_variant,
                'progress_percent': progress_percent,
                'cache_size': len(self.variant_cache)
            }
            
            with open(self.checkpoint_file, 'w') as f:
                json.dump(checkpoint, f, indent=2)
            
            # Save cache periodically
            if processed_rows_count % 1000 == 0:
                self._save_cache()
                
        except Exception as e:
            print(f"✗ Error saving checkpoint: {e}")
    
    def _save_cache(self):
        """Save variant cache to disk"""
        try:
            with open(self.cache_file, 'wb') as f:
                pickle.dump(self.variant_cache, f)
        except Exception as e:
            print(f"✗ Error saving cache: {e}")
    
    def _save_results(self, dataframe, indices):
        """Save partial results to disk"""
        try:
            partial = {}
            for idx in indices:
                partial[idx] = dataframe.at[idx, 'hgvs_genome']
            
            with open(self.results_file, 'wb') as f:
                pickle.dump(partial, f)
        except Exception as e:
            print(f"✗ Error saving results: {e}")
    
    def _show_progress(self, processed, total, cache_hits, phase, 
                      current_variant=None, batch_info=None):
        """Display progress in Jupyter Notebook"""
        clear_output(wait=True)
        
        # Progress bar
        progress_percent = (processed / total * 100) if total > 0 else 0
        bar_length = 50
        filled_length = int(bar_length * processed // total)
        bar = '█' * filled_length + '░' * (bar_length - filled_length)
        
        # Create progress display
        html_content = f"""
        <div style="font-family: monospace; padding: 10px; border: 1px solid #ccc; border-radius: 5px; background: #f9f9f9;">
            <h3 style="margin-top: 0;">HGVS Processing - Phase {phase}</h3>
            
            <div style="margin: 10px 0;">
                <div style="display: flex; justify-content: space-between; margin-bottom: 5px;">
                    <span>Progress: {progress_percent:.1f}%</span>
                    <span>{processed:,} / {total:,}</span>
                </div>
                <div style="height: 20px; background: #ddd; border-radius: 10px; overflow: hidden;">
                    <div style="height: 100%; width: {progress_percent}%; background: #4CAF50; transition: width 0.3s;"></div>
                </div>
            </div>
            
            <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 10px; margin: 15px 0;">
                <div style="background: #e8f5e9; padding: 10px; border-radius: 5px;">
                    <strong>Cache Statistics</strong><br>
                    Hits: {cache_hits:,}<br>
                    Cache Size: {len(self.variant_cache):,}
                </div>
                
                <div style="background: #e3f2fd; padding: 10px; border-radius: 5px;">
                    <strong>Processing Info</strong><br>
                    Phase: {phase}<br>
                    {f'Batch: {batch_info}' if batch_info else ''}
                </div>
            </div>
            
            {f'<div style="background: #fff3e0; padding: 10px; border-radius: 5px; margin-top: 10px; font-size: 12px;"><strong>Current Variant:</strong><br>{current_variant[:80]}{"..." if len(current_variant) > 80 else ""}</div>' if current_variant else ''}
            
            <div style="margin-top: 15px; padding-top: 10px; border-top: 1px solid #ddd; font-size: 12px; color: #666;">
                Checkpoint saved automatically. Close and reopen notebook to resume.
            </div>
        </div>
        """
        
        display(HTML(html_content))
        
        # Also print to console for logging
        print(f"Phase {phase}: {processed:,}/{total:,} ({progress_percent:.1f}%) | "
              f"Cache: {cache_hits:,} hits | Cache size: {len(self.variant_cache):,}")
        if current_variant:
            print(f"Current: {current_variant[:100]}{'...' if len(current_variant) > 100 else ''}")
    
    def process_variant(self, variant_string):
        """Process a single variant string with error handling"""
        if not isinstance(variant_string, str) or pd.isna(variant_string):
            return None
        
        # Check cache first
        if variant_string in self.variant_cache:
            return self.variant_cache[variant_string]
        
        # Validate format
        if ':c.' not in variant_string:
            result = f"Error: Invalid format, expected ':c.' in {variant_string[:50]}..."
            self.variant_cache[variant_string] = result
            return result
        
        try:
            transcript_id = variant_string.split(':c.')[0]
            rest = variant_string.split(':c.')[1]
            
            # Parse multiple variants
            variants = []
            if '[' in rest and ']' in rest:
                inside_brackets = rest.split('[')[1].split(']')[0]
                variant_parts = [part.strip() for part in inside_brackets.split(';') if part.strip()]
                variants = [f"{transcript_id}:c.{part}" for part in variant_parts]
            elif ',' in rest and '[' not in rest:
                parts = [part.strip() for part in rest.split(',') if part.strip()]
                variants = [f"{transcript_id}:c.{part}" for part in parts]
            else:
                variants = [variant_string]
            
            genomic_components = []
            accession = None
            
            for var in variants:
                try:
                    parsed = self.hp.parse_hgvs_variant(var)
                    genomic = self.am.c_to_g(parsed)
                    genomic_str = str(genomic)
                    
                    if not accession:
                        accession = genomic_str.split(':g.')[0]
                    
                    position = genomic_str.split(':g.')[1]
                    genomic_components.append(position)
                    
                    del parsed
                    del genomic
                    
                except Exception as e:
                    result = f"Error in {var[:50]}: {str(e)[:100]}"
                    self.variant_cache[variant_string] = result
                    return result
            
            # Format output
            if not genomic_components:
                result = None
            elif len(genomic_components) == 1:
                result = f"{accession}:g.{genomic_components[0]}"
            else:
                result = f"{accession}:g.[{';'.join(genomic_components)}]"
            
            self.variant_cache[variant_string] = result
            return result
            
        except Exception as e:
            result = f"Error: {str(e)[:100]}"
            self.variant_cache[variant_string] = result
            return result
    
    def process_dataframe(self, dataframe, variant_column='hgvs_transcript', 
                         result_column='hgvs_genome', batch_size=1000,
                         resume=True, show_progress=True):
        """
        Process variants in a DataFrame with resume capability
        
        Parameters:
        -----------
        dataframe : pandas DataFrame
            DataFrame containing variant strings
        variant_column : str
            Column name containing HGVS transcript variants
        result_column : str
            Column name to store genomic results
        batch_size : int
            Number of variants to process before saving checkpoint
        resume : bool
            Whether to resume from checkpoint
        show_progress : bool
            Whether to display progress in notebook
            
        Returns:
        --------
        pandas DataFrame
            Updated DataFrame with genomic variants
        """
        
        print("="*70)
        print("HGVS TRANSCRIPT TO GENOMIC CONVERSION")
        print("="*70)
        
        # Validate input
        if variant_column not in dataframe.columns:
            raise ValueError(f"Column '{variant_column}' not found in DataFrame")
        
        # Initialize result column
        if result_column not in dataframe.columns:
            dataframe[result_column] = None
        
        # Filter variants of interest
        mask = dataframe[variant_column].astype(str).str.startswith('NM_000157.4')
        variant_series = dataframe.loc[mask, variant_column]
        total_rows = len(variant_series)
        
        if total_rows == 0:
            print("No variants starting with NM_000157.4 found")
            return dataframe
        
        # Get unique variants
        duplicate_counts = variant_series.value_counts()
        unique_variants_count = len(duplicate_counts)
        
        print(f"Total variants to process: {total_rows:,}")
        print(f"Unique variants: {unique_variants_count:,}")
        print(f"Duplication factor: {total_rows/unique_variants_count:.2f}x")
        print(f"Cache already contains: {len(self.variant_cache):,} variants")
        print("-"*70)
        
        # Determine resume point
        if resume and self.checkpoint:
            print("🔄 RESUMING FROM CHECKPOINT")
            phase = self.checkpoint.get('phase', 1)
            processed_unique_count = self.checkpoint.get('processed_unique_count', 0)
            processed_rows_count = self.checkpoint.get('processed_rows_count', 0)
            
            # We need to reconstruct which variants have been processed
            # This is simplified - in production you'd want to save the actual list
            processed_unique_set = set()
            
            # Filter remaining variants (simplified approach)
            unique_variants = duplicate_counts.index.tolist()
            
            # Since we don't know which exact ones were processed,
            # we'll start from the beginning but skip based on count
            start_index = min(processed_unique_count, len(unique_variants))
            unique_variants = unique_variants[start_index:]
            
            # For rows, we need to track which indices are done
            # We'll assume the first N rows were processed
            indices_to_process = variant_series.index.tolist()
            start_row_index = min(processed_rows_count, len(indices_to_process))
            indices_to_process = indices_to_process[start_row_index:]
            
            print(f"Already processed approximately: {processed_unique_count:,} unique variants")
            print(f"Already processed approximately: {processed_rows_count:,} rows")
            print(f"Remaining: {len(unique_variants):,} unique variants")
            print(f"Remaining: {len(indices_to_process):,} rows")
            
        else:
            print("🚀 STARTING NEW PROCESSING")
            phase = 1
            processed_unique_count = 0
            processed_rows_count = 0
            unique_variants = duplicate_counts.index.tolist()
            indices_to_process = variant_series.index.tolist()
            
            # Clear old checkpoint if starting fresh
            if os.path.exists(self.checkpoint_file):
                os.remove(self.checkpoint_file)
        
        # PHASE 1: Process unique variants
        if phase == 1 and unique_variants:
            print(f"\n📊 PHASE 1: Processing {len(unique_variants):,} unique variants")
            
            cache_hits = 0
            start_time = time.time()
            
            for i, variant_string in enumerate(unique_variants):
                # Process variant
                result = self.process_variant(variant_string)
                processed_unique_count += 1
                
                # Show progress
                if show_progress and (i % 10 == 0 or i == len(unique_variants) - 1):
                    self._show_progress(
                        processed_unique_count, 
                        unique_variants_count,
                        cache_hits,
                        phase=1,
                        current_variant=variant_string,
                        batch_info=f"{i+1}/{len(unique_variants)}"
                    )
                
                # Save checkpoint every batch_size
                if i % batch_size == 0 or i == len(unique_variants) - 1:
                    self._save_checkpoint(
                        phase=1,
                        processed_unique_count=processed_unique_count,
                        processed_rows_count=processed_rows_count,
                        total_unique=unique_variants_count,
                        total_rows=total_rows,
                        current_variant=variant_string
                    )
                    self._save_cache()
                    
                    if i > 0:
                        elapsed = time.time() - start_time
                        variants_per_second = (i + 1) / elapsed
                        remaining = len(unique_variants) - (i + 1)
                        eta = remaining / variants_per_second if variants_per_second > 0 else 0
                        
                        print(f"  Speed: {variants_per_second:.1f} variants/sec")
                        print(f"  ETA: {eta/60:.1f} minutes remaining")
            
            print(f"\n✅ PHASE 1 COMPLETE")
            print(f"  Processed {processed_unique_count:,} unique variants")
            print(f"  Time elapsed: {(time.time() - start_time)/60:.1f} minutes")
            
            # Move to phase 2
            phase = 2
            self._save_checkpoint(
                phase=2,
                processed_unique_count=processed_unique_count,
                processed_rows_count=processed_rows_count,
                total_unique=unique_variants_count,
                total_rows=total_rows,
                current_variant="Starting phase 2"
            )
        
        # PHASE 2: Apply cached results to all rows
        if phase == 2 and indices_to_process:
            print(f"\n📊 PHASE 2: Applying results to {len(indices_to_process):,} rows")
            
            cache_hits = 0
            start_time = time.time()
            results_indices = []
            
            for i, idx in enumerate(indices_to_process):
                variant_string = dataframe.at[idx, variant_column]
                
                # Get result from cache
                if variant_string in self.variant_cache:
                    result = self.variant_cache[variant_string]
                    cache_hits += 1
                else:
                    # Should not happen, but process if needed
                    result = self.process_variant(variant_string)
                
                # Update DataFrame
                dataframe.at[idx, result_column] = result
                processed_rows_count += 1
                results_indices.append(idx)
                
                # Show progress
                if show_progress and (i % 100 == 0 or i == len(indices_to_process) - 1):
                    self._show_progress(
                        processed_rows_count,
                        total_rows,
                        cache_hits,
                        phase=2,
                        current_variant=None,
                        batch_info=f"{i+1}/{len(indices_to_process)}"
                    )
                
                # Save checkpoint and results periodically
                if i % batch_size == 0 or i == len(indices_to_process) - 1:
                    self._save_checkpoint(
                        phase=2,
                        processed_unique_count=processed_unique_count,
                        processed_rows_count=processed_rows_count,
                        total_unique=unique_variants_count,
                        total_rows=total_rows,
                        current_variant=f"Row {i+1}/{len(indices_to_process)}"
                    )
                    
                    # Save partial results
                    self._save_results(dataframe, results_indices)
                    results_indices = []
                    
                    if i > 0:
                        elapsed = time.time() - start_time
                        rows_per_second = (i + 1) / elapsed
                        remaining = len(indices_to_process) - (i + 1)
                        eta = remaining / rows_per_second if rows_per_second > 0 else 0
                        
                        print(f"  Speed: {rows_per_second:.1f} rows/sec")
                        print(f"  ETA: {eta/60:.1f} minutes remaining")
                
                # Clean up memory
                if i % 5000 == 0:
                    gc.collect()
            
            print(f"\n✅ PHASE 2 COMPLETE")
            print(f"  Processed {processed_rows_count:,} rows")
            print(f"  Cache hits: {cache_hits:,} ({cache_hits/total_rows*100:.1f}%)")
            print(f"  Time elapsed: {(time.time() - start_time)/60:.1f} minutes")
            
            # Clean up checkpoint files
            if os.path.exists(self.checkpoint_file):
                os.remove(self.checkpoint_file)
                print(f"  Removed checkpoint file")
            
            if os.path.exists(self.results_file):
                os.remove(self.results_file)
                print(f"  Removed partial results file")
            
            # Save final cache
            self._save_cache()
        
        # Print summary statistics
        print("\n" + "="*70)
        print("📈 PROCESSING SUMMARY")
        print("="*70)
        
        # Count results
        results_series = dataframe.loc[mask, result_column]
        error_mask = results_series.astype(str).str.startswith('Error')
        valid_mask = results_series.notna() & ~error_mask
        
        error_count = error_mask.sum()
        valid_count = valid_mask.sum()
        null_count = results_series.isna().sum()
        
        print(f"Total variants processed: {total_rows:,}")
        print(f"Valid results: {valid_count:,} ({valid_count/total_rows*100:.1f}%)")
        print(f"Errors: {error_count:,} ({error_count/total_rows*100:.1f}%)")
        print(f"Null results: {null_count:,} ({null_count/total_rows*100:.1f}%)")
        print(f"Final cache size: {len(self.variant_cache):,} variants")
        
        # Show sample of errors if any
        if error_count > 0:
            print(f"\n❌ Sample errors:")
            error_samples = results_series[error_mask].head(5)
            for variant, error in zip(variant_series[error_mask].head(5), error_samples):
                print(f"  {variant[:60]}... → {error}")
        
        # Show sample of valid results
        if valid_count > 0:
            print(f"\n✅ Sample valid results:")
            valid_samples = results_series[valid_mask].head(5)
            for variant, result in zip(variant_series[valid_mask].head(5), valid_samples):
                print(f"  {variant[:60]}... → {result[:60]}...")
        
        print("\n💾 Cache saved for future runs")
        print("🔄 Run again to resume if interrupted")
        
        return dataframe
    
    def get_stats(self):
        """Get processing statistics"""
        stats = {
            'cache_size': len(self.variant_cache),
            'checkpoint_exists': os.path.exists(self.checkpoint_file),
            'checkpoint_dir': self.checkpoint_dir
        }
        
        if os.path.exists(self.checkpoint_file):
            try:
                with open(self.checkpoint_file, 'r') as f:
                    checkpoint = json.load(f)
                    stats.update({
                        'phase': checkpoint.get('phase'),
                        'processed_unique_count': checkpoint.get('processed_unique_count', 0),
                        'processed_rows_count': checkpoint.get('processed_rows_count', 0),
                        'timestamp': checkpoint.get('timestamp'),
                        'progress_percent': checkpoint.get('progress_percent', 0)
                    })
            except:
                pass
        
        return stats
    
    def clear_checkpoint(self):
        """Clear all checkpoint files (start fresh)"""
        files = [self.checkpoint_file, self.cache_file, 
                self.progress_file, self.results_file]
        
        for file in files:
            if os.path.exists(file):
                os.remove(file)
                print(f"Removed: {file}")
        
        self.checkpoint = None
        self.variant_cache = {}
        print("✅ All checkpoints cleared. Ready for fresh start.")

In [31]:
# Cell 3: Usage Example and Initialization
# Initialize processor
processor = HGVSProcessor(checkpoint_dir='./hgvs_checkpoints')

# Check current status
stats = processor.get_stats()
print("Current Status:")
for key, value in stats.items():
    print(f"  {key}: {value}")

Initializing HGVS components (attempt 1/5)...
✓ HGVS components initialized successfully
✓ Loaded 27 cached variants
✓ Found checkpoint from: 2025-12-13 18:12:17
  Phase: 1
  Processed unique variants: 0
  Processed rows: 0
  Progress: 0.0%
✓ HGVS Processor initialized
  Checkpoint directory: ./hgvs_checkpoints
  Cache size: 27 variants
Current Status:
  cache_size: 27
  checkpoint_exists: True
  checkpoint_dir: ./hgvs_checkpoints
  phase: 1
  processed_unique_count: 4
  processed_rows_count: 0
  timestamp: 2025-12-13 18:12:17
  progress_percent: 0.0


In [ ]:
# Cell 4: Process Your Variants
# Load your data
# Assuming you have a DataFrame called 'variants'
variants = pd.read_csv('gba1_variants_with_hgvs_transcript.csv', keep_default_na=False)
variants.head()

# Process with resume capability
# This will resume from where it left off if you close and reopen the notebook
if 'variants' in locals() and isinstance(variants, pd.DataFrame):
    print("Starting processing...")
    variants = processor.process_dataframe(
        variants,
        variant_column='hgvs_transcript',  # Change to your column name
        result_column='hgvs_genome',       # Column to store results
        batch_size=500,                     # Save checkpoint every 500 variants
        resume=True,                       # Resume from checkpoint
        show_progress=True                 # Show progress in notebook
    )
else:
    print("Please load your variants DataFrame first")


Phase 1: 1,655/17,189 (9.6%) | Cache: 0 hits | Cache size: 1,655
Current: NM_000157.4:c.[544C>A;546G>T]


In [ ]:
variants = pd.read_csv('gba1_variants_with_hgvs_transcript.csv', keep_default_na=False)
variants.head()

,library,barcode,variant_call_support,number_of_indels,codon_substitutions,aa_substitutions,n_codon_substitutions,n_aa_substitutions,variant_class,aa_substitutions_c,hgvs_transcript
0,GBA1,AAGCGGACTGCACGTCGTCTTTTTTTTTTT,1,0,,,0,0,wildtype,wildtype,
1,GBA1,ACCATGTATCGACCCTATCATTTTTTTTTT,1,0,CGC78CAG,R78Q,1,1,1 nonsynonymous,R78Q,NM_000157.4:c.[233G>A;234C>G]
2,GBA1,TAAGTCGCTGCGCGGATGTCGTTTTTTTTT,13,0,,,0,0,wildtype,wildtype,
3,GBA1,AGGCAAAACTGCTACTTGCCTGTTTTTTTT,8,0,AGC47TGG,S47W,1,1,1 nonsynonymous,S47W,NM_000157.4:c.[139A>T;141C>G]
4,GBA1,TTGTTTAGCTGTCAACCCCCCGTTTTTTTT,44,0,,,0,0,wildtype,wildtype,


In [ ]:
# Cell 5: Updated Quick Check Functions
def check_status():
    """Check current processing status"""
    stats = processor.get_stats()
    
    if stats['checkpoint_exists']:
        print(f"📌 CHECKPOINT FOUND")
        print(f"   Phase: {stats.get('phase')}")
        print(f"   Progress: {stats.get('progress_percent', 0):.1f}%")
        print(f"   Processed unique variants: {stats.get('processed_unique_count', 0):,}")
        print(f"   Processed rows: {stats.get('processed_rows_count', 0):,}")
        print(f"   Last update: {stats.get('timestamp')}")
        print(f"\n   Cache contains: {stats['cache_size']:,} variants")
        
        resume_option = input("\nResume processing? (y/n): ")
        if resume_option.lower() == 'y':
            return 'resume'
        else:
            return 'stop'
    else:
        print("No checkpoint found. Ready for new processing.")
        return 'new'

In [ ]:
# Cell 6: Resume After Interruption
# If you close your laptop or lose connection, just run this cell to resume

print("Checking for previous processing...")
status = check_status()

if status == 'resume':
    print("\nResuming processing...")
    # Re-initialize processor (will load checkpoint)
    processor = HGVSProcessor(checkpoint_dir='./hgvs_checkpoints')
    
    # Continue processing (make sure 'variants' DataFrame is loaded)
    if 'variants' in locals():
        variants = processor.process_dataframe(
            variants,
            variant_column='hgvs_transcript',
            result_column='hgvs_genome',
            resume=True,
            show_progress=True
        )
    else:
        print("Error: 'variants' DataFrame not found. Please load your data first.")
elif status == 'new':
    print("Starting new processing...")
    # Make sure your 'variants' DataFrame is loaded
    if 'variants' in locals():
        processor = HGVSProcessor(checkpoint_dir='./hgvs_checkpoints')
        variants = processor.process_dataframe(
            variants,
            variant_column='hgvs_transcript',
            result_column='hgvs_genome',
            resume=False,  # Start fresh
            show_progress=True
        )
    else:
        print("Please load your variants DataFrame first")
else:
    print("Processing stopped.")

Checking for previous processing...
📌 CHECKPOINT FOUND
   Phase: 1
   Progress: 0.0%
   Processed unique variants: 1
   Processed rows: 0
   Last update: 2025-12-13 17:42:18

   Cache contains: 1 variants
Processing stopped.


In [ ]:
# Cell 7: Save Results
# After processing completes, save your results

if 'variants' in locals():
    # Save to CSV
    variants.to_csv('processed_variants_with_genomic.csv', index=False)
    print("Results saved to 'processed_variants_with_genomic.csv'")
    
    # Export cache for future use
    processor._save_cache()
    print("Cache saved for future processing")
    
    # Show summary
    total = len(variants)
    valid = variants['hgvs_genome'].notna().sum()
    errors = variants['hgvs_genome'].astype(str).str.startswith('Error').sum()
    
    print(f"\nFinal Summary:")
    print(f"  Total variants: {total:,}")
    print(f"  Valid conversions: {valid:,} ({valid/total*100:.1f}%)")
    print(f"  Errors: {errors:,} ({errors/total*100:.1f}%)")
else:
    print("No variants DataFrame found. Process your data first.")

No variants DataFrame found. Process your data first.
